In [ ]:
# conda activate genomic_tools

import pandas as pd
from pyfaidx import Fasta

In [ ]:
# Sequence-based domain detection (run HMMER/ELM directly on the exon peptide) sidesteps coordinates — a match is a match regardless of numbering. 
# But position-based cross-referencing (UniProt features, mapping full-protein InterProScan hits onto exons, isoform diffs) genuinely needs a correct, shared Met1.

# Goal: narrow down the space of exons being considered for runningw ith InterProScan

### First load in gene descriptions from Ensembl and Uniprot. 
In addition to selecting exons with the most significant cell type associations, I want to prioritize exons from genes with certain function, e.g. receptors, ion channels, synaptic proteins, etc.

In [ ]:
ensembl_info = pd.read_csv("data/ensembl_gene_info.csv")
uniprot_info = pd.read_csv("data/gene_uniprot_info.csv")

In [ ]:
interesting_genes = set(ensembl_info['Gene Symbol']).intersection(set(uniprot_info['Gene Symbol']))

In [ ]:
signif_exons = pd.read_csv("data/ctype_exons/annotated/All_GABAergic_exons_annotated.csv")

signif_exons = signif_exons.rename(columns={signif_exons.columns[0]: "event"})

### For each cell type, subset to top exons, and exons from genes with biologically interesting functions.

### Then, for exons compatible with multiple transcripts, select highest scoring transcript

In [ ]:
def score_transcript(x):
    score = 0
    if "MANE_Select" in str(x['tag']):
        score += 1
    if "appris_principal" in str(x['tag']):
        score += 1
    if "basic" in str(x['tag']):
        score += 1
    if "CCDS" in str(x['tag']):
        score += 1
    if "GENCODE_Primary" in str(x['tag']):
        score += 1
    if x['overlap_type'] == "fully_coding":
        score += 1
    return score

In [ ]:
transcript_list = []

for file in os.listdir("data/ctype_exons/annotated"):
    signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
    signif_exons = signif_exons.rename(columns={signif_exons.columns[0]: "event"})
    
    # subset to protein coding trancripts, since we're interested in protein domains
    # note: will look into non-coding transcripts later, but for now, focus on the ones we can annotate with InterProScan
    signif_coding_exons = signif_exons[signif_exons['transcript_type'] == "protein_coding"]
    
    # subset to top exons AND/OR exons from genes with relevant biological functions
    signif_coding_exons = signif_coding_exons[signif_coding_exons['Gene'].isin(interesting_genes)]
    
    # then, for each exon, keep the transcript with the highest score (based on GENCODE tags)
    signif_coding_exons['transcript_score'] = signif_coding_exons.apply(score_transcript, axis=1)
    idx = signif_coding_exons.groupby("event")['transcript_score'].idxmax()
    signif_coding_exons_max = signif_coding_exons.loc[idx]
    
    transcript_list.extend(signif_coding_exons_max['transcript_id'].tolist())
    
transcripts = list(set(transcript_list))  # unique transcripts 

### Get amino acid sequence for transcripts identified in previous step; save in FASTA format for input to InterProScan

In [ ]:
proteins = Fasta("/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v49.pc_translations.fa")

# Build a dict keyed by ENST
protein_by_transcript = {}
for key in proteins.keys():
    parts = key.split("|")
    enst_versioned = parts[1].split(".")[0]
    protein_by_transcript[enst_versioned] = str(proteins[key])

In [ ]:
modified_transcript_products = {}
transcript_log = []

with open("data/proteins.fa", "w") as f:
    for key in proteins.keys():
        enst = key.split("|")[1].split(".")[0]
        if enst in transcripts:
            transcript_log.append(enst)
            seq = str(proteins[key]).rstrip("*")
            if "X" in seq:
                # Track where in the sequence the X was 
                modified_transcript_products[enst] = [i for i, c in enumerate(seq) if c == "X"]
                seq = seq.replace("X", "")
            f.write(f">{enst}\n{seq}\n")